In [1]:
import argparse
from pathlib import Path
from typing import Dict
import os
import json

import torch
import torch.distributed
from finetuning import CheckpointManager

from unakin.llm.adapters.llm_adapter import LlmAdapter
from unakin.llm.dialogue.base import DialogueModel
from unakin.llm.models.utils import ModelFactory
from unakin.llm.tokenizers import LlamaTokenizer
from unakin.llm.tokenizers.base import TokenizerBase

/home/fsuser/miniconda3/envs/ncs-stable/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Optional

REFERENCE_MAP = {
  "13B": {
    "model": {
        "type": "llama",
        "args": {
          "base": "code-llama-13b-instruct"
        }
      },
      "tokenizer": {
        "type": "llama",
        "path": "/mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer.model"
      }
  },
  "34B": {
    "model": {
        "type": "llama",
        "args": {
          "base": "code-llama-34b-instruct"
        }
      },
      "tokenizer": {
        "type": "llama",
        "path": "/mnt/raid0/llamas/codellama/CodeLlama-34b-Instruct/tokenizer.model"
      }
  }
}

def get_ckpt(path: str, skip_path: bool = True) -> Dict:
  if skip_path:
    return {}

  ckpt = torch.load(f"{path}/ckpt.pt", map_location="cpu")
  return ckpt

def _get_dev_model(reference_type: str = "13B") -> DialogueModel:
  model, tokenizer = ModelFactory.from_json(REFERENCE_MAP[reference_type])

  model.enable_xops(enable=True)
  model.to(torch.cuda.current_device())

  # model.to(torch.cuda.current_device())

  return DialogueModel(model=model, tokenizer=tokenizer)


def get_dialogue_model_from_ckpt(
    ckpt_root: str, ckpt_name: str, ckpt_step: str,
    reference_type: Optional[str] = None, batch_size: int = 1) -> DialogueModel:
  
  path = Path(ckpt_root) / ckpt_name / ckpt_step / "ckpt.pth"

  if reference_type is not None:
    model = _get_dev_model()
    model.model.register_cache_kv(max_bsz=batch_size, max_seq_len=4096)
    return model

  ckpt_manager = CheckpointManager(path)
  
  # model
  model, tokenizer = ckpt_manager.init_model_from_ckpt(path=path, root_path=ckpt_root)
  model: LlmAdapter

  # move model to device
  model = model.to(torch.cuda.current_device())
  # model = model.half()

  # merge adaptation
  model.merge_adaptation()
  model.register_cache_kv(max_bsz=batch_size, max_seq_len=8192)

  # create the dialogue model
  dialogue_model = DialogueModel(model=model, tokenizer=tokenizer)

  return dialogue_model

def read_json_lines(path: str):
  assert path.endswith(".jsonl")
  with open(path, "r") as f:
    lines = f.readlines()
    lines = [json.loads(line) for line in lines]
    lines = [line["task"] for line in lines]
    return lines

def read_system_prompt(path: str):
  if not path:
    return None

  with open(path, "r") as f:
    return f.read().strip()
  
def get_pad_dialogue(batch: list[list], target_size: int = 16, add_system_prompt: bool = True):
  pad = [{"role": "user", "content": "blank"}]
  if add_system_prompt:
    pad = [{"role": "system", "content": "blank"}] + pad
  
  for _ in range(target_size - len(batch)):
    batch.append(pad)
    
  return batch  


def init_torch(args):
  if args.world_size > 1:
    init_torch_distributed(args)
    rank = torch.distributed.get_rank()
    device = torch.device(f"cuda:{rank}")
  else:
    device = torch.device(f"cuda:{args.device_id}")
  torch.cuda.set_device(device)
  torch.set_default_dtype(torch.float16)

def init_torch_distributed(args):
  init_torch(args)
  torch.distributed.init_process_group(
    backend="nccl",
    init_method="env://",
  )

def make_dialogue_message(content: str, system_prompt: str | None):
  user_message = {"role": "user", "content": content}
  if system_prompt is None:
    return [user_message]
  
  system_message = {"role": "system", "content": system_prompt}
  return [system_message, user_message]



In [5]:
import argparse

args = argparse.Namespace(
  world_size=1,
  device_id=0,
  ckpt_root="/mnt/raid0/",
  ckpt_name="checkpoints/bigmix_ft_fim_rev_f5tkbnmt",
  ckpt_step="000704",
  reference_type=None,
)

In [6]:
init_torch(args)
dialogue_model = get_dialogue_model_from_ckpt(
      ckpt_root=args.ckpt_root,
      ckpt_name=args.ckpt_name,
      ckpt_step=args.ckpt_step,
      reference_type=args.reference_type,
  )

/mnt/raid0/llamas/llama_base/code-llama-13b-instruct.pth code-llama-13b-instruct
/mnt/raid0/llamas/llama_base/code-llama-13b-instruct.pth
Allocated virtual model in 0.024105310440063477s
Loaded virtual model in 0.006663799285888672s
XxxXXXXXXXXXXX Tokenizer path: /mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer.model
Loading tokenizer from /mnt/raid0/llamas/codellama/CodeLlama-13b-Instruct/tokenizer.model
INFO: Registering cache of size 1 x 8192


In [5]:
# import pandas as ps
data = []
for name, param in dialogue_model.model.named_parameters():
  print(name, param.size(), param.norm().item(), param.mean().item(), param.std().item(), param.dtype)
  data.append([name, param.size(), param.norm().item(), param.mean().item(), param.std().item(), name.split(".")[-1]])

# df = ps.DataFrame(data, columns=["name", "size", "norm", "mean", "std", "layer"])

base_model.tok_embeddings.weight torch.Size([32016, 5120]) 278.0 8.535385131835938e-05 0.021728515625 torch.bfloat16
base_model.layers.0.attention.wq.B torch.Size([5120, 256]) 1144.0 -0.00063323974609375 1.0 torch.bfloat16
base_model.layers.0.attention.wq.A torch.Size([256, 5120]) 0.07470703125 1.367880031466484e-09 6.532669067382812e-05 torch.bfloat16
base_model.layers.0.attention.wq.adapted_linear.weight torch.Size([5120, 5120]) 65.5 -1.257285475730896e-07 0.01275634765625 torch.bfloat16
base_model.layers.0.attention.wk.B torch.Size([5120, 256]) 1144.0 2.0265579223632812e-05 1.0 torch.bfloat16
base_model.layers.0.attention.wk.A torch.Size([256, 5120]) 0.095703125 -1.885928213596344e-08 8.344650268554688e-05 torch.bfloat16
base_model.layers.0.attention.wk.adapted_linear.weight torch.Size([5120, 5120]) 69.5 1.1026859283447266e-06 0.01361083984375 torch.bfloat16
base_model.layers.0.attention.wv.B torch.Size([5120, 256]) 1144.0 -0.0015411376953125 1.0 torch.bfloat16
base_model.layers.0.a

In [10]:
df

,name,size,norm,mean,std,layer
0,base_model.tok_embeddings.weight,"(32016, 5120)",277.750000,8.559227e-05,0.021698,weight
1,base_model.layers.0.attention.wq.B,"(5120, 256)",1145.000000,-6.318092e-04,1.000000,B
2,base_model.layers.0.attention.wq.A,"(256, 5120)",0.074829,0.000000e+00,0.000065,A
3,base_model.layers.0.attention.wq.adapted_linea...,"(5120, 5120)",65.250000,-1.192093e-07,0.012749,weight
4,base_model.layers.0.attention.wk.B,"(5120, 256)",1144.000000,2.026558e-05,0.999512,B
...,...,...,...,...,...,...
918,base_model.layers.39.feed_forward.w3.adapted_l...,"(13824, 5120)",219.500000,3.516674e-06,0.026093,weight
919,base_model.layers.39.attention_norm.weight,"(5120,)",26.484375,3.681641e-01,0.039581,weight
920,base_model.layers.39.ffn_norm.weight,"(5120,)",26.859375,3.747559e-01,0.022064,weight
921,base_model.norm.weight,"(5120,)",62.312500,8.691406e-01,0.060394,weight


In [7]:
import pprint

code = """
```csharp
using UnityEngine;
using System.Collections;
using Assets.Scripts;

[RequireComponent(typeof(Rigidbody2D))]
public class Bird : MonoBehaviour
{

    // Use this for initialization
    void Start()
    {
        //trailrenderer is not visible until we throw the bird
        GetComponent<TrailRenderer>().enabled = false;
        GetComponent<TrailRenderer>().sortingLayerName = "Foreground";
        //no gravity at first
        GetComponent<Rigidbody2D>().isKinematic = true;
        //make the collider bigger to allow for easy touching
        GetComponent<CircleCollider2D>().radius = Constants.BirdColliderRadiusBig;
        State = BirdState.BeforeThrown;
    }



    void FixedUpdate()
    {
        // if we've thrown the bird and its speed is very small
        if (State == BirdState.Thrown &&
            GetComponent<Rigidbody2D>().velocity.sqrMagnitude <= Constants.MinVelocity)
        {
            // destroy the bird after 2 seconds
            StartCoroutine(DestroyAfter(2));
        }

    }


    public void OnThrow()
    {
        //play the sound
        GetComponent<AudioSource>().Play();
        //show the trail renderer
        GetComponent<TrailRenderer>().enabled = true;
        //allow for gravity forces
        GetComponent<Rigidbody2D>().isKinematic = false;
        //make the collider normal size
        GetComponent<CircleCollider2D>().radius = Constants.BirdColliderRadiusNormal;
        State = BirdState.Thrown;
    }

    IEnumerator DestroyAfter(float seconds)
    {
        yield return new WaitForSeconds(seconds);
        Destroy(gameObject);
    }

    public BirdState State
    {
        get;
        private set;
    }
}
```"""

system_prompt = f"You are a helpfull engineering assistant. You design problem descriptions and instructions for generating code snniptes. You use only natural language and do not respond with code."
instruction = f"Using the provided code snippet as a reference, create a comprehensive problem description that includes necessary details. Ensure that the problem is clearly defined and highlights any constraints. The problem description should be detailed enough for an engineer to understand and implement a solution:\n```\n{code}```"
# instruction = f"Given the following code, generate the appropriate instructions for the problem:\n{code}\n"
# instruction = f"Generate an instruction for the following code:\n{code}\n"

def role_dialogue_message(content: str, role: str):
  return {"role": role, "content": content}

# system_0 = role_dialogue_message(
#     content="You are a helpfull engineering assistant. You design problem descriptions and instructions for generating code snniptes. You use only natural language and do not respond with code.", role="system"
#   )

instruction_1 = """Tell me a story about a lumberjack."""
message_1 = role_dialogue_message(
    content=instruction, role="user"
  )

# response_2 = {
#     "content": """Once upon a time, there was a lumberjack named Jack who lived in a small cabin in the woods. He spent his days chopping down trees and transporting them to the sawmill. Jack was a skilled lumberjack and took pride in his work.

# One day, while Jack was chopping down a particularly large tree, he heard a strange noise coming from the woods. It sounded like someone was crying. Jack went to investigate and found a little girl lost in the woods. She was shivering and scared, and couldn't find her way back home.

# Jack took the little girl back to his cabin and gave her a warm meal and a place to stay for the night. He also made sure she was safe and sound. The next day, he helped her find her way home, and she thanked him for his kindness.

# From that day on, Jack and the little girl became good friends. She would often visit him in the woods, and they would spend hours talking and playing together. Jack was happy to have a friend and was grateful for the opportunity to help someone in need.

# Years went by, and Jack grew old and retired from his job as a lumberjack. But he never forgot about the little girl he had helped so many years ago. He would often think back on their adventures together, and smile at the memories.

# One day, when Jack was very old and frail, he fell ill and was bedridden. The little girl came to visit him and brought him a big basket of food and drinks. She sat by his bedside and held his hand, and Jack felt grateful for her kindness and friendship.

# As Jack lay there, he knew that his time on earth was coming to an end. He looked at the little girl and smiled, and she smiled back. Then, Jack closed his eyes and took his last breath, surrounded by the love and kindness of his dear friend.

# The little girl buried Jack in the woods, where he had spent so many happy years chopping down trees and living his life. She placed a big stone on top of his grave, and wrote his name and the date of his birth and death on it. She also left a note for anyone who found his grave, telling them the story of Jack and the little girl, and how they had become the best of friends.

# Years went by, and the little girl grew old and passed away. But her memory lived on, and whenever anyone visited Jack's grave, they would hear her story and be reminded of the kindness and friendship that had brought them together."""
# }
# message_2 = role_dialogue_message(
#     content=response_2["content"], role="assistant"
#   )

# instruciton_3 = """Where did Jack live?"""

# message_3 = role_dialogue_message(
#     content=instruciton_3, role="user"
#   )

# response_4 = {
#     "content": """Maria took a chest filled with gold from the enemy."""}

# message_4 = role_dialogue_message(
#     content=response_4["content"], role="assistant"
#   )

# instruciton_5 = """Who was the enemy?"""


# message_5 = role_dialogue_message(
#     content=instruciton_5, role="user"
#   )


message_list = [message_1] #, message_2, message_3] #, message_4, message_5] # message_6, message_7]


prompt_prediction = dialogue_model.chat_completion(
  [message_list], temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.0)
  
# pprint.pprint(prompt_prediction)
print(prompt_prediction[0]["generation"]["content"].strip())

Generating 1 sequences of max length 2601 on cuda:0


  0%|          | 0/2048 [00:00<?, ?it/s]

100%|██████████| 2048/2048 [00:47<00:00, 43.13it/s]

You are given a game in which birds are thrown from a bird feeder. The birds fly around the screen and can be caught by the player.

Your task is to implement the functionality of catching birds.

The game consists of the following components:

1. Birds

The birds are represented by a circle. The circle is placed at the center of the bird's sprite. The circle has a radius of 0.5. The circle is used for collision detection.

The birds have the following states:

* BeforeThrown - the bird is not thrown yet
* Thrown - the bird is in flight
* Caught - the bird is caught by the player

The bird can be thrown in the following way:

1. The player presses the "throw" button.
2. The bird is placed at the center of the bird feeder.
3. The bird is given an initial velocity.
4. The bird is set to the "Thrown" state.

The bird can be caught by the player in the following way:

1. The player presses the "catch" button.
2. The bird is set to the "Caught" state.

The bird can be destroyed in the follo

In [6]:
# prediction = prompt_prediction[0]["generation"]["content"]
# example = f"""
# ### INPUT CODE
# {code}

# ### GENERATED DESCRIPTION
# {prediction}
# """

In [12]:
import pandas as pd
from typing import NamedTuple

class Sample(NamedTuple):
  description: str
  code: str

class DiscriminativeSample(NamedTuple):
  generated_instruction: str
  code: str

df = pd.read_json("all_data.json", orient="records")
df = df[df["Codes_Count"] < 2]

samples = []
for idx, row in df.iterrows():
  code = row["Codes"][0]
  code = code.replace("\r\n", "\n")
  code = f"```csharp\n{code}\n```"
  samples.append(Sample(description=row["Prompt"], code=code))

discriminative_samples = []


gen_instr_prompt = """Generate a high quality instruction for the provided csharp code. Do not write lists or steps as instructions. Write a high level instruction that could have been solved with the provided code. Do not reference the code or blocks of it. \n{code}"""

user_instructions = [
"""
Below is an instruction from an user and a candidate answer. Evaluate whether or
not the answer is a good example of how AI Assistant should respond to the user’s
instruction:
1: It means the answer is incomplete, vague, off-topic, controversial, or not
exactly what the user asked for. For example, some content seems missing, numbered
list does not start from the beginning, the opening sentence repeats user’s question.
Or the response is from another person’s perspective with their personal experience
(e.g. taken from blog posts), or looks like an answer from a forum. Or it contains
promotional text, navigation text, or other irrelevant information.
2: It means the answer addresses most of the asks from the user. It does not
directly address the user’s question. For example, it only provides a high-level
methodology instead of the exact solution to user’s question.
3: It means the answer is helpful but not written by an AI Assistant. It addresses
all the basic asks from the user. It is complete and self contained with the
drawback that the response is not written from an AI assistant’s perspective, but
from other people’s perspective. The content looks like an excerpt from a blog post,
web page, or web search results. For example, it contains personal experience or
opinion, mentions comments section, or share on social media, etc.
4: It means the answer is written from an AI assistant’s perspective with a
clear focus of addressing the instruction. It provide a complete, clear, and
comprehensive response to user’s question or instruction without missing or
irrelevant information. It is well organized, self-contained, and written in a
helpful tone. It has minor room for improvement, e.g. more concise and focused.
5: It means it is a perfect answer from an AI Assistant. It has a clear focus on
being a helpful AI Assistant, where the response looks like intentionally written
to address the user’s question or instruction without any irrelevant sentences. The
answer provides high quality content, demonstrating expert knowledge in the area, is
very well written, logical, easy-to-follow, engaging and insightful.
{instruction}
{code}
Please first provide a reasoning you used to derive the rating score. Explain your reasoning step by step. Be very critical and detailed.
""",
"""On the next line, extract only the rating number and write it."""
]

# init the prompt turns (instructions)
for s in samples[:10]:

  message_list = []
  code = s.code

  # generate the instruction
  user_instr = gen_instr_prompt.format(code=code)
  message_list.append(role_dialogue_message(content=user_instr, role="user"))

  description = dialogue_model.chat_completion(
    [message_list], temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.)

  instruction = description[0]["generation"]["content"].strip()

  pprint.pprint(description)
  print(s.description)
  print("*" * 80)

  message_list = []

  for i in range(10):

    message_list = []

    for user_instr in user_instructions:

      if "{instruction}" in user_instr and "{code}" in user_instr:
        user_instr = user_instr.format(instruction=instruction, code=code)

      message_list.append(role_dialogue_message(content=user_instr, role="user"))

      description = dialogue_model.chat_completion(
        [message_list], temperature=0.6, top_p=0.9, max_gen_len=2048, repeat_penalty=1.)
      
      discriminative_samples.append(DiscriminativeSample(
        generated_instruction=description[0]["generation"]["content"],
        code=code
      ))
      
      prev_response = description[0]["generation"]["content"]
      message_list.append(role_dialogue_message(content=prev_response, role="assistant"))
      
      pprint.pprint(description)
      print(s.description)
      print("*" * 80)

  break


Generating 1 sequences of max length 3196 on cuda:0


  0%|          | 0/2048 [00:00<?, ?it/s]

 53%|█████▎    | 1092/2048 [00:25<00:22, 43.17it/s]


[{'generation': {'content': ' ```csharp\n'
                            'using System.Collections;\n'
                            'using System.Collections.Generic;\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class PauseUI : MonoBehaviour\n'
                            '{\n'
                            '    [SerializeField]\n'
                            '    Button _pauseButton;\n'
                            '    [SerializeField]\n'
                            '    Button _resumeButton;\n'
                            '    [SerializeField]\n'
                            '    TextMeshProUGUI _pauseText;\n'
                            '    [SerializeField]\n'
                            '    const string PAUSE_TEXT = "PAUSED";\n'
                            '    [SerializeField]\n'
                            '  

 53%|█████▎    | 1092/2048 [00:26<00:22, 41.60it/s]


...

[{'generation': {'content': ' ```csharp\n'
                            'using System.Collections;\n'
                            'using System.Collections.Generic;\n'
                            'using UnityEngine;\n'
                            'using UnityEngine.UI;\n'
                            'using TMPro;\n'
                            '\n'
                            'public class PauseUI : MonoBehaviour\n'
                            '{\n'
                            '    [SerializeField]\n'
                            '    Button _pauseButton;\n'
                            '    [SerializeField]\n'
                            '    Button _resumeButton;\n'
                            '    [SerializeField]\n'
                            '    TextMeshProUGUI _pauseText;\n'
                            '    [SerializeField]\n'
                            '    const string PAUSE_TEXT = "PAUSED";\n'
                            '    [SerializeField]\n'
                            '  

  0%|          | 7/2048 [00:00<03:13, 10.55it/s]


[{'generation': {'content': ' ```\n4\n``` ', 'role': 'assistant'}}]
Create a simple UI script for two buttons that can pause and resume the game
********************************************************************************
Generating 1 sequences of max length 4771 on cuda:0


 30%|██▉       | 611/2048 [00:14<00:34, 41.38it/s]


KeyboardInterrupt: 

In [4]:
try:
  del discriminator_dialogue_model
except Exception as e:
  pass

args_discriminator = argparse.Namespace(
  world_size=1,
  device_id=2,
  ckpt_root="/home/ted/workdir/ncs/ml-research/recipes",
  ckpt_name="checkpoints/l4_instr_fw_20M",
  ckpt_step="001280",
  reference_type="34B",
)

init_torch(args_discriminator)

discriminator_dialogue_model = get_dialogue_model_from_ckpt(
      ckpt_root=args_discriminator.ckpt_root,
      ckpt_name=args_discriminator.ckpt_name,
      ckpt_step=args_discriminator.ckpt_step,
      reference_type=args_discriminator.reference_type,
)

discriminator_dialogue_model.model.register_cache_kv(max_bsz=1, max_seq_len=8192)

/home/ted/fsx/llama_base/code-llama-13b-instruct.pth code-llama-13b-instruct
/home/ted/fsx/llama_base/code-llama-13b-instruct.pth
Allocated virtual model in 0.04566311836242676s
Loaded virtual model in 0.011732339859008789s
INFO: Registering cache of size 1 x 4096
INFO: Registering cache of size 1 x 8192
